In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("..")

In [3]:
import torch
import json
import numpy as np
from transformers import AutoTokenizer
from tqdm.auto import tqdm
from datasets import load_dataset
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.float
device   = 'cuda'
model_id = "Qwen/QwQ-32B"

In [4]:
from pathlib import Path

cur_dir = Path(".").absolute()


def load_dataset_from_file(domain_name, task_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/qwq-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)

In [5]:
task_name = "plan_generation_po"
eval_results = [
    load_dataset_from_file(domain_name, task_name)["instances"] for domain_name in [
        "blocksworld_mystery",
    ]
]
eval_results = [{x["dataset_idx"]: x for x in er} for er in eval_results]

In [6]:

tokenizer = initialize_tokenizer(model_id)

In [7]:
dataset = load_dataset(f"dmitriihook/qwq-32b-planning-mystery-24k")["train"]

In [8]:
DOMAIN_PHRASES = {
    "mystery_1": {
        "actions": {
            "attack": "attack",
            "succumb": "succumb",
            "overcome": "overcome",
            "feast": "feast"
        },
        "predicates": {
            "planet": "planet",
            "province": "province",
            "harmony": "harmony",
            "craves": "craves",
            "pain": "pain"
        }
    },
    "mystery_2": {
        "actions": {
            "attack": "illuminate",
            "succumb": "silence",
            "overcome": "distill",
            "feast": "divest"
        },
        "predicates": {
            "planet": "aura",
            "province": "essence",
            "harmony": "nexus",
            "craves": "harmonizes",
            "pain": "pulse"
        }
    },
}

In [9]:
def extract_all_phrase_positions(tokens, phrase, tokenizer, cot_only=True, min_pos=None):
    """Find end of the phrase token positions"""
    tokens = tokens.squeeze()

    phrase_tokens = [
        tokenizer.encode(" " + phrase),
        tokenizer.encode(" " + phrase.capitalize()),
        tokenizer.encode("\n" + phrase)[1:],
        tokenizer.encode("\n" + phrase.capitalize())[1:],
        tokenizer.encode("\n\n" + phrase)[1:],
        tokenizer.encode("\n\n" + phrase.capitalize())[1:],
    ]

    positions = set()

    if cot_only:
        start_pos = torch.where(tokens == 151667)[0]
        start_mask = torch.arange(tokens.shape[0]) >= start_pos

    for phts in phrase_tokens:
        presence_mask = torch.ones_like(tokens)
        if cot_only:
            presence_mask = presence_mask * start_mask

        for i, t in enumerate(phts):
            presence_mask = presence_mask * (tokens == t)[i:]
            presence_mask = presence_mask[:-1]

        for p in (torch.where(presence_mask)[0]).tolist():
            if min_pos is not None and p < min_pos:
                continue
            positions.add(
                tuple([p-1, p + len(phts)])
            )        
    
    return sorted(list(set(positions)))

In [10]:
import nest_asyncio

nest_asyncio.apply()

In [11]:
import asyncio
import io
import os

from PIL import Image
import requests
import sglang as sgl

from sglang.srt.conversation import chat_templates
from sglang.test.test_utils import is_in_ci
from sglang.utils import async_stream_and_merge, stream_and_merge

if is_in_ci():
    import patch
else:
    import nest_asyncio

    nest_asyncio.apply()


llm = sgl.Engine(model_path=model_id)

INFO 04-13 21:10:24 __init__.py:190] Automatically detected platform cuda.
INFO 04-13 21:10:35 __init__.py:190] Automatically detected platform cuda.
INFO 04-13 21:10:36 __init__.py:190] Automatically detected platform cuda.
[2025-04-13 21:10:41,198] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


Loading safetensors checkpoint shards:   0% Completed | 0/14 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   7% Completed | 1/14 [00:00<00:09,  1.33it/s]
Loading safetensors checkpoint shards:  14% Completed | 2/14 [00:01<00:09,  1.29it/s]
Loading safetensors checkpoint shards:  21% Completed | 3/14 [00:02<00:08,  1.27it/s]
Loading safetensors checkpoint shards:  29% Completed | 4/14 [00:03<00:07,  1.27it/s]
Loading safetensors checkpoint shards:  36% Completed | 5/14 [00:03<00:05,  1.56it/s]
Loading safetensors checkpoint shards:  43% Completed | 6/14 [00:04<00:05,  1.46it/s]
Loading safetensors checkpoint shards:  50% Completed | 7/14 [00:05<00:05,  1.39it/s]
Loading safetensors checkpoint shards:  57% Completed | 8/14 [00:05<00:04,  1.34it/s]
Loading safetensors checkpoint shards:  64% Completed | 9/14 [00:06<00:03,  1.32it/s]
Loading safetensors checkpoint shards:  71% Completed | 10/14 [00:07<00:03,  1.30it/s]
Loading safetensors checkpoint shards:  79% Completed | 11/14

In [13]:
?llm

Type:           Engine
String form:    <sglang.srt.entrypoints.engine.Engine object at 0x7fba5e8a9250>
File:           ~/openr1/lib/python3.11/site-packages/sglang/srt/entrypoints/engine.py
Docstring:     
The entry point to the inference engine.

- The engine consists of three components:
    1. TokenizerManager: Tokenizes the requests and sends them to the scheduler.
    2. Scheduler (subprocess): Receives requests from the Tokenizer Manager, schedules batches, forwards them, and sends the output tokens to the Detokenizer Manager.
    3. DetokenizerManager (subprocess): Detokenizes the output tokens and sends the result back to the Tokenizer Manager.

Note:
1. The HTTP server, Engine, and TokenizerManager both run in the main process.
2. Inter-process communication is done through ICP (each process uses a different port) via the ZMQ library.
Init docstring:
The arguments of this function is the same as `sglang/srt/server_args.py::ServerArgs`.
Please refer to `ServerArgs` for the docu